# Health-Gait Manifest Loader

This notebook turns the Health-Gait silhouette manifest into a PyTorch `Dataset` and `DataLoader`.

The manifest is a CSV index. Each row describes one walking trial and points to a directory of frame images. The dataset reads one row, loads a contiguous frame window from that row's `frame_dir`, and returns a tensor shaped `[T, C, H, W]`.

The `DataLoader` then stacks several samples into `[B, T, C, H, W]`, which is the shape a video model usually expects before any model-specific rearrangement.

## Imports

The notebook imports the reusable Health&Gait loader package instead of defining a second notebook-local dataset class.

- The short path bootstrap lets the notebook run from either the repo root or `notebooks/`.
- `HealthGaitManifestDataset` is the low-level PyTorch dataset.
- `cody_jepa.data` provides path, manifest-preview, dataset, and DataLoader builder functions.
- Matplotlib is only for the visual sanity check at the end.
- PyTorch is used for the training handoff stub.


In [1]:
from pathlib import Path
import json
import sys

import torch
from IPython.display import Image as DisplayImage, display

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / "src").exists() else NOTEBOOK_CWD.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from cody_jepa.data import (
    HealthGaitLoaderConfig,
    HealthGaitManifestDataset,
    build_healthgait_datasets_from_config,
    build_healthgait_loaders_from_config,
    find_repo_root,
    healthgait_manifest_path,
    preview_manifest,
    run_healthgait_motion_diagnostics,
    summarize_healthgait_manifest,
    write_healthgait_dummy_probe_exports,
    write_healthgait_metadata_summary,
)


## Path Setup

The manifest stores paths like `data/healthgait/raw/...`, which are relative to the repo root. Jupyter may start either in the repo root or inside `notebooks/`, so this cell finds the repo root once and anchors every manifest path to it.

In [ ]:
REPO_ROOT = find_repo_root()
MANIFEST_CSV = healthgait_manifest_path(REPO_ROOT)
LOADER_CONFIG = HealthGaitLoaderConfig(
    manifest_csv=MANIFEST_CSV,
    repo_root=REPO_ROOT,
    split="train",
)

print(f"repo root: {REPO_ROOT}")
print(f"manifest:  {MANIFEST_CSV}")
print(f"exists:    {MANIFEST_CSV.exists()}")
print("loader config:")
print(json.dumps(LOADER_CONFIG.as_dict(), indent=2, sort_keys=True))


## Manifest Contract

Before writing the dataset class, inspect the CSV contract. The important columns are:

- `frame_dir`: directory containing ordered image frames for one trial.
- `num_frames`: how many frames the manifest builder counted in that directory.
- `split`: whether the trial belongs to `train` or `val`.
- `subject_id`: used for diagnostics, not as a self-supervised training label.

A good PyTorch dataset should treat each manifest row as one sample source, not as one frame.

In [ ]:
preview_rows = preview_manifest(MANIFEST_CSV, n=3)

print("columns:", list(preview_rows[0].keys()) if preview_rows else [])
print("first row:")
display(preview_rows[0])

DIAGNOSTICS_DIR = REPO_ROOT / "data" / "healthgait" / "diagnostics"
manifest_summary = summarize_healthgait_manifest(
    MANIFEST_CSV,
    repo_root=REPO_ROOT,
    clip_length=LOADER_CONFIG.clip_length,
)
summary_paths = write_healthgait_metadata_summary(
    manifest_summary,
    DIAGNOSTICS_DIR,
    "healthgait_manifest_summary",
)
print(json.dumps(manifest_summary, indent=2, sort_keys=True))
print(f"saved: {summary_paths['json']}")
print(f"saved: {summary_paths['csv']}")


## Dataset Class

The dataset implementation now lives in `src/dataset.py`, so training scripts and notebooks use the same code path.

During `__init__`, it validates the whole manifest before building samples: required columns, `train`/`val` split values, existing `frame_dir` directories, `num_frames` consistency with listed frame files, enough frames for the configured clip length, nonempty train and validation splits, and no subject overlap between train and validation.

After validation, it builds a lightweight list of samples. Each sample stores the row metadata and the sorted frame paths. This avoids crawling the filesystem again every time `__getitem__` runs.

During `__getitem__`, it loads only the frames needed for one clip window. That keeps memory use low: the full dataset is not loaded into RAM.


In [ ]:
print(f"dataset class: {HealthGaitManifestDataset.__module__}.{HealthGaitManifestDataset.__name__}")


## Dataset Policy

`LOADER_CONFIG` records the manifest path, repo root, split, clip length, image size, channel count, seed, window policy, validation policy, and DataLoader settings in one checkpoint-friendly object.

With the default `train_random_val_center` window policy, the train split can yield different temporal windows across epochs while validation always picks the center window. Sampling remains deterministic for a given config seed, `sequence_id`, epoch, and split, so it stays reproducible even when a training script uses multiple DataLoader workers.

In [ ]:
diagnostic_epoch = 0

train_ds, val_ds = build_healthgait_datasets_from_config(LOADER_CONFIG)
train_ds.set_epoch(diagnostic_epoch)

print(f"train clips: {len(train_ds)}")
print(f"val clips:   {len(val_ds)}")


## Create DataLoaders

The dataset returns one sample. The `DataLoader` handles batching.

Start with `num_workers=0` in notebooks because it gives clearer error messages. In a training script, increasing `num_workers` can improve throughput once the loader is correct.

In [ ]:
train_loader, val_loader = build_healthgait_loaders_from_config(LOADER_CONFIG)


## Inspect One Batch

The key output is `video`. Its shape should be `[B, T, C, H, W]`:

- `B`: batch size.
- `T`: frames per clip.
- `C`: channels, which is `1` for grayscale silhouettes.
- `H`, `W`: resized frame height and width.

The pixel range should be `[0, 1]` because `_load_frame` divides the original `uint8` pixels by `255.0`.

Each clip also carries source metadata: `sequence_id`, `split`, `modality`, `subject_id`, `gait_system`, `trial`, `window_start`, `frame_indices`, and `num_frames`.

In [ ]:
batch = next(iter(train_loader))
video = batch["video"]

print(f"video shape: {tuple(video.shape)}")
print(f"dtype:       {video.dtype}")
print(f"min/max:     {float(video.min()):.3f} / {float(video.max()):.3f}")
print(f"sequence:    {batch['sequence_id'][0]}")
print(f"split:       {batch['split'][0]}")
print(f"modality:    {batch['modality'][0]}")
print(f"subject:     {batch['subject_id'][0]}")
print(f"gait system: {batch['gait_system'][0]}")
print(f"trial:       {batch['trial'][0]}")
print(f"window:      {int(batch['window_start'][0])}")
print(f"frames:      {[int(frame[0]) for frame in batch['frame_indices']]}")
print(f"num frames:  {int(batch['num_frames'][0])}")

## Motion Diagnostics

Run a deterministic seeded sample from each split and save reusable diagnostics under `data/healthgait/diagnostics/`. The report includes a clip contact sheet, mean absolute frame-difference maps, a motion-energy histogram, per-sample CSV statistics, and compact low/high-motion examples.

These are explicit reporting calls; constructing the datasets does not write files.

In [ ]:
motion_diagnostics = run_healthgait_motion_diagnostics(
    {"train": train_ds, "val": val_ds},
    DIAGNOSTICS_DIR,
    samples_per_split=8,
    seed=LOADER_CONFIG.seed,
    epoch=diagnostic_epoch,
)
print(json.dumps({
    "sample_count": motion_diagnostics["sample_count"],
    "low_motion_examples": motion_diagnostics["low_motion_examples"],
    "high_motion_examples": motion_diagnostics["high_motion_examples"],
}, indent=2, sort_keys=True))

for artifact_name in ("contact_sheet", "diff_maps", "histogram"):
    artifact_path = motion_diagnostics["artifacts"][artifact_name]
    print(f"{artifact_name}: {artifact_path}")
    display(DisplayImage(filename=str(artifact_path)))


## Dummy Probe Export

Write a placeholder probe export with the same stream-latent schema Week 8 probes will consume. The current vectors are deterministic clip statistics; later model exports can replace the values while keeping the column contract stable.

In [ ]:
PROBE_EXPORT_DIR = REPO_ROOT / "data" / "healthgait" / "probe_exports"
probe_export = write_healthgait_dummy_probe_exports(
    {"train": train_ds, "val": val_ds},
    PROBE_EXPORT_DIR,
    stem="healthgait_dummy_probe_exports",
    latent_dim=8,
    epoch=diagnostic_epoch,
)
print(f"rows:  {probe_export['row_count']}")
print(f"saved: {probe_export['csv']}")
print("columns:")
probe_export["fieldnames"]


## Training Loop Shape Stub

This is only a handoff point to a model, not a full training loop.

For self-supervised CoDy-JEPA training, the model should consume `batch["video"]`. Keep `subject_id` and `trial` for diagnostics and debugging. Do not treat `subject_id` as a training label in the self-supervised objective.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

metadata_keys = (
    "sequence_id",
    "split",
    "subject_id",
    "gait_system",
    "trial",
    "window_start",
)

for epoch in range(1):
    train_loader.dataset.set_epoch(epoch)

    for batch in train_loader:
        metadata_for_logging = {key: batch[key] for key in metadata_keys}
        video = batch["video"].to(device)  # JEPA loss path: [B, T, C, H, W]

        print(f"send this tensor to the model: {tuple(video.shape)} on {device}")
        print(f"metadata stays outside the loss path: {metadata_for_logging['sequence_id'][0]}")
        break